# AgentCore workshop — building the refund agent in the console

Our company sells electronics online, and our support inbox is drowning in the same
message, over and over: *"I want to refund my order."* Every one of them means a person
looking up the order, checking whether it's refundable, and replying.

You've been asked to build an **AI refund agent** to take that off the team's plate. Not
a demo — something we can actually put in front of customers. So we'll build it the way
it grows in real life: start with the simplest agent that works, then add one AgentCore
feature at a time, each time because the agent ran into something it couldn't handle.

**Everything is done in the AWS console — there's no code to run.** This notebook is
your step-by-step guide: what to click, what to paste, and how to check each step worked.

| Step | What you add | AgentCore feature |
|---|---|---|
| 1 | A refund agent you can chat with | **Harness** → runs on **Runtime** |
| 2 | An orders Lambda (the "database" lookup) | — |
| 3 | Expose the Lambda as tools | **Gateway** |
| 4 | Give the tools to the agent | Harness + Gateway |

```
 chat ──► agent (Harness on Runtime) ──MCP──► Gateway ──► Lambda ──► mock ORDERS
          └──── step 1 ────┘                  └────────── steps 2–4 ──────────┘
```

**You need:** an AWS account with Bedrock access to a Claude model in your region.
Create everything in the **same region**.

## 1. Deploy the agent with a Harness

Let's start with the smallest thing that could possibly help a customer: an agent you
can talk to. No database, no tools — just a model with a clear job description, running
somewhere real. We want it **deployed**, not on a laptop, because that's where it has to
live eventually.

A **Harness** is a declarative agent: you pick a model, write a system prompt, and
attach tools — AgentCore runs it on **Runtime** for you. No code, no container.

### Create it

1. **Bedrock AgentCore → Harness → Create**.
2. **Name:** `refund-agent`
3. **Model:** a Claude model available in your region.
4. **System prompt** (use this from the start; later steps only add tools):

   ```
   You are the refund assistant for our online store. Help the customer check their orders
   and request refunds.
   - Always use the tools to look up orders and transactions; never guess an order's
     status, amount, or refund state.
   - If the customer doesn't give an order id, use find_orders to locate it.
   - Only delivered orders can be refunded.
   - Be concise and friendly.
   ```
5. **Tools:** none yet.
6. **Execution role:** let the console create one.
7. Create, and wait until it's ready.

### Check: chat with it

Open the Harness's **test chat** and try:

| Prompt | What to look for |
|---|---|
| `Hi, what can you help me with?` | It answers in role — the model, prompt and Runtime work |
| `I'm alice. I want to refund my keyboard.` | It has **no tools**, so it should say it can't look orders up. Did it make something up instead? |

> 🔎 If the agent invented an order status or a refund, that's the problem the next steps
> fix: an agent can only be as truthful as the tools it's given.

**✅ Done when:** the test chat replies, and the agent doesn't claim to have found your order.

## 2. The tool: an orders Lambda

So we have an agent — and it's polite, it understands what the customer wants, and it
can't do a single useful thing about it. Ask it about your keyboard and the best it can
say is *"I can't look that up."* (And if it said anything else, it was guessing.)

To actually help with a refund, the agent has to do what a support person does first:
**look the order up**. Which order does the customer mean? Has it been delivered? What
was paid, and has it already been refunded? All of that lives in our order database.

So we'll give the agent **tools** — functions it can decide to call when it needs a fact
it doesn't have. The first tool is a lookup into the order database. For now the
"database" is a small set of fake orders inside a Lambda function; later it becomes a
real table, and the agent won't notice the difference.

### What the code does

- **`ORDERS`** — a static dict standing in for the database (all values are fake).
  Each order has a status, a payment `transaction`, and a `refund` if one exists.
- **Two tools**, because customers rarely know their order id:
  - `find_orders(customer_id, item_query?)` — "refund my keyboard" → which order is that?
  - `get_order_transaction(order_id)` — status, amount, payment, existing refund.
- **`lambda_handler`** — the Gateway calls *one* Lambda for every tool and tells it
  which tool via `context.client_context` (`"<target>___<tool>"`). The handler routes
  on the part after `___`.

> 🔎 Look at `get_order_transaction`: it returns an order to **anyone who asks for its id**.
> Keep that in mind — we come back to it in the identity step.

### Create it

1. **Lambda → Create function** → *Author from scratch*
   - **Name:** `orders-tool` · **Runtime:** Python 3.12
2. In the code editor, replace the contents of `lambda_function.py` with the code below
   and click **Deploy**.

<details>
<summary><b>lambda_function.py</b> — click to expand, then copy</summary>

```python
"""Order & transaction lookup, as an AgentCore Gateway Lambda tool.

The agent calls this when a customer says "I want to refund my order ..." --
first to find which order they mean, then to read its transaction details.

The "database" is a static dict below (a mock -- all values are fake). Later it
becomes a DynamoDB table; the tool signatures stay the same.

The Gateway routes each MCP tool call here and passes the tool name in
context.client_context.custom["bedrockAgentCoreToolName"] as "<target>___<tool>".
The tool arguments arrive as the `event` dict.
"""
import json
import logging
from typing import Any, Dict

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Mock database: order_id -> order + its payment transaction. All values are fake.
ORDERS: Dict[str, Dict[str, Any]] = {
    "A-1001": {
        "customer_id": "alice",
        "item": "Wireless mouse",
        "order_date": "2026-09-12",
        "status": "delivered",
        "transaction": {
            "transaction_id": "txn_7f3a91",
            "amount": 49.00,
            "currency": "USD",
            "payment_method": "card ending 4242",
            "paid_at": "2026-09-12T10:14:00Z",
        },
        "refund": None,
    },
    "A-1002": {
        "customer_id": "alice",
        "item": "Mechanical keyboard",
        "order_date": "2026-09-25",
        "status": "shipped",
        "transaction": {
            "transaction_id": "txn_8c21d4",
            "amount": 120.00,
            "currency": "USD",
            "payment_method": "card ending 4242",
            "paid_at": "2026-09-25T16:40:00Z",
        },
        "refund": None,
    },
    "A-1003": {
        "customer_id": "alice",
        "item": "Laptop stand",
        "order_date": "2026-08-30",
        "status": "refunded",
        "transaction": {
            "transaction_id": "txn_2b77e0",
            "amount": 35.00,
            "currency": "USD",
            "payment_method": "card ending 4242",
            "paid_at": "2026-08-30T09:02:00Z",
        },
        "refund": {"refund_id": "rf_2b77e0", "amount": 35.00, "refunded_at": "2026-09-04T11:00:00Z"},
    },
    "B-2001": {
        "customer_id": "bob",
        "item": "USB-C cable",
        "order_date": "2026-09-18",
        "status": "delivered",
        "transaction": {
            "transaction_id": "txn_5e90aa",
            "amount": 15.00,
            "currency": "USD",
            "payment_method": "card ending 1111",
            "paid_at": "2026-09-18T13:27:00Z",
        },
        "refund": None,
    },
    "B-2002": {
        "customer_id": "bob",
        "item": "4K monitor",
        "order_date": "2026-09-20",
        "status": "delivered",
        "transaction": {
            "transaction_id": "txn_c4f612",
            "amount": 300.00,
            "currency": "USD",
            "payment_method": "card ending 1111",
            "paid_at": "2026-09-20T08:55:00Z",
        },
        "refund": None,
    },
}

TOOLS = {}


def tool(name: str):
    """Register a function as a gateway tool."""
    def decorator(func):
        TOOLS[name] = func
        return func
    return decorator


def lambda_handler(event: Dict[str, Any], context) -> Dict[str, Any]:
    """Route a gateway MCP tool call to the matching function."""
    # The Lambda console's Test button can't set client_context -- for console tests,
    # put the tool name in the test event as "tool_name" instead.
    custom = context.client_context.custom if context.client_context else {}
    extended = custom.get("bedrockAgentCoreToolName") or event.pop("tool_name", "")
    tool_name = extended.split("___", 1)[1] if "___" in extended else extended
    logger.info(json.dumps({"tool": tool_name, "args": event}))

    handler = TOOLS.get(tool_name)
    if not handler:
        return {"statusCode": 400, "body": json.dumps({"error": f"Unknown tool: {tool_name}"})}
    try:
        return {"statusCode": 200, "body": json.dumps({"result": handler(event)})}
    except Exception as exc:  # surface the message to the model, don't leak a stack
        logger.exception("tool failed")
        return {"statusCode": 500, "body": json.dumps({"error": str(exc)})}


@tool("find_orders")
def find_orders(event: Dict[str, Any]) -> Dict[str, Any]:
    """Find a customer's orders, optionally filtered by item name.

    Used when the customer doesn't give an order id ("refund my keyboard").

    Args:
        customer_id: the customer whose orders to search, e.g. "alice"
        item_query: optional, case-insensitive text to match in the item name
    """
    customer_id = event.get("customer_id")
    if not customer_id:
        return {"error": "Missing required parameter: customer_id."}
    query = (event.get("item_query") or "").lower()

    matches = [
        {"order_id": oid, "item": o["item"], "status": o["status"], "order_date": o["order_date"],
         "amount": o["transaction"]["amount"], "currency": o["transaction"]["currency"]}
        for oid, o in ORDERS.items()
        if o["customer_id"] == customer_id and query in o["item"].lower()
    ]
    return {"orders": matches}


@tool("get_order_transaction")
def get_order_transaction(event: Dict[str, Any]) -> Dict[str, Any]:
    """Get one order's full details: status, payment transaction, and any refund.

    Args:
        order_id: the order to look up, e.g. "A-1001"
    """
    # WORKSHOP NOTE: this trusts any order_id and returns whoever owns it -- there
    # is no ownership check. That's deliberate: it's the hole the identity step
    # (failure mode 3) closes.
    order_id = event.get("order_id")
    if not order_id:
        return {"error": "Missing required parameter: order_id."}
    order = ORDERS.get(order_id)
    if not order:
        return {"error": f"No order {order_id} found."}
    return {"order_id": order_id, **order}


if __name__ == "__main__":
    # Self-check: routing + both tools, no AWS needed.
    class _Ctx:
        class client_context:
            custom = {"bedrockAgentCoreToolName": "orders___find_orders"}

    found = lambda_handler({"customer_id": "alice", "item_query": "keyboard"}, _Ctx())
    assert found["statusCode"] == 200 and "A-1002" in found["body"], found
    _Ctx.client_context.custom["bedrockAgentCoreToolName"] = "orders___get_order_transaction"
    txn = lambda_handler({"order_id": "A-1001"}, _Ctx())
    assert txn["statusCode"] == 200 and "txn_7f3a91" in txn["body"], txn
    missing = lambda_handler({"order_id": "Z-9999"}, _Ctx())
    assert "No order Z-9999" in missing["body"], missing
    _Ctx.client_context.custom["bedrockAgentCoreToolName"] = "orders___nope"
    assert lambda_handler({}, _Ctx())["statusCode"] == 400

    class _ConsoleCtx:  # what a Lambda console test looks like: no client_context
        client_context = None

    console = lambda_handler({"tool_name": "find_orders", "customer_id": "alice"}, _ConsoleCtx())
    assert console["statusCode"] == 200 and "A-1001" in console["body"], console
    print("orders lambda self-check OK: routing + both tools + console test events")
```

</details>

### Check: test it in the console

The Gateway passes the tool name in a way a console test can't, so for tests the code
also accepts it as `tool_name` in the test event.

1. **Test** tab → **Create new event**, name `find-keyboard`, event JSON:

   ```json
   {"tool_name": "find_orders", "customer_id": "alice", "item_query": "keyboard"}
   ```
2. Click **Test**. The response `body` should contain order **`A-1002`**.
3. Optional — a second event, `get-a1001`:

   ```json
   {"tool_name": "get_order_transaction", "order_id": "A-1001"}
   ```
   The `body` should contain transaction **`txn_7f3a91`**.

**✅ Done when:** both tests return `statusCode: 200` with the expected order.

## 3. Expose it through an AgentCore Gateway

The lookup works — but only when *we* call it from the console. The agent has no idea
it exists, what it's called, or what to pass it.

We could wire the Lambda straight into the agent. But this won't stay one tool:
next come refunds, shipping, maybe a CRM — and other agents will want them too. Instead
of every agent wiring up every tool itself, we put the tools behind one front door that
handles discovery, auth and routing.

That front door is the **AgentCore Gateway**. It turns your Lambda into an **MCP
server** any agent can connect to.

To do that, the Gateway needs a **tool schema**. It's worth a minute, because the schema
is read by the **model**, not by the Lambda. For each tool it says:

- **`name`** — what the agent calls. It must match a tool the Lambda knows.
- **`description`** — *when* to use it. The model chooses tools by reading this, so it's
  effectively part of your prompt.
- **`inputSchema`** — the arguments, and which ones are required.

<details>
<summary><b>Tool schema</b> — click to expand, then copy</summary>

```json
[
  {
    "name": "find_orders",
    "description": "Find a customer's orders, optionally filtered by item name. Use when the customer doesn't give an order id.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "customer_id": {
          "type": "string",
          "description": "The customer whose orders to search, e.g. \"alice\"."
        },
        "item_query": {
          "type": "string",
          "description": "Optional text to match in the item name, e.g. \"keyboard\"."
        }
      },
      "required": ["customer_id"]
    }
  },
  {
    "name": "get_order_transaction",
    "description": "Get one order's status, payment transaction, and any existing refund.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string",
          "description": "The order identifier, e.g. \"A-1001\"."
        }
      },
      "required": ["order_id"]
    }
  }
]
```

</details>

### Create it

1. **Bedrock AgentCore → Gateways → Create gateway**. Name it, e.g. `refund-agent-gateway`.
2. **Inbound auth: IAM.** The agent signs its requests with its own IAM role — no tokens
   yet. (Quick start may default to a Cognito/JWT login; that's what the identity step
   adds later. For now choose IAM.)
3. **Service role:** let the console create one.
4. **Target:** type **Lambda**, pick `orders-tool`, choose an **inline** schema and paste
   the tool schema above.
5. Create.

> **If the target fails with** *"Gateway service is not authorized to perform AssumeRole
> on Gateway role"* — the console just created that role and IAM hasn't caught up.
> **Wait a minute and retry creating the target.** If it still fails, open the role in
> **IAM → Trust relationships** and check it allows `bedrock-agentcore.amazonaws.com`,
> and that the region in its `aws:SourceArn` condition matches your gateway's region.

### Check

On the Gateway's page:

- Status is **Ready**, inbound auth is **IAM**.
- Your target is **Ready** and lists two tools: `find_orders`, `get_order_transaction`.
- Note the **Gateway ARN** — you need it in the next step.

**✅ Done when:** the gateway and its target are both Ready.

## 4. Give the tools to the agent

We now have an agent that can talk, and a set of tools behind a Gateway. Time to
introduce them: we connect the agent to the Gateway, and make sure the agent is
**allowed** to use it. Then we go back to the customer from step 1 — the one with the
keyboard — and see if we can help them this time.

### Attach the Gateway

1. Open your `refund-agent` **Harness** → edit **Tools** → add your **Gateway**. Save.
   The system prompt from step 1 already tells it to use `find_orders` — no change needed.

### Allow the agent to call the Gateway

With IAM inbound auth, the Gateway only accepts calls from roles that are allowed to
make them. In **IAM → Roles →** the Harness's **execution role** → **Add permissions →
Create inline policy → JSON**, paste this, putting in your Gateway ARN:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "bedrock-agentcore:InvokeGateway",
      "Resource": "<YOUR_GATEWAY_ARN>"
    }
  ]
}
```

> **If the test chat fails with** *"Failed to load tool … Failed to start MCP client …
> 403 Forbidden"* — the Gateway refused the agent's first call. Either the Gateway's
> inbound auth is JWT/Cognito (the agent sends no token → use IAM), or the Harness role
> is missing the policy above. IAM changes take a minute; reload the chat after.

### Check: chat with it again

Re-run step 1's keyboard prompt — now it should find the order. The agent has **no login
yet**, so tell it who you are in the prompt:

| Prompt | Expected tool calls | What it shows |
|---|---|---|
| `I'm alice. I want to refund my keyboard.` | `find_orders` → `get_order_transaction("A-1002")` | Finds the order from a description; A-1002 is **shipped**, so not refundable yet |
| `I'm alice. Show me the payment details for order A-1001.` | `get_order_transaction` | Direct lookup |
| `I'm alice. I want a refund for my laptop stand.` | `find_orders` → `get_order_transaction("A-1003")` | Already refunded — the agent should say so |
| `I'm alice. What orders do I have?` | `find_orders` | Listing |

Confirm the tools really ran:

- In the Harness's **trace** for the request, find the tool calls. Open a tool result —
  that text is **exactly what the model read**.
- **Lambda → `orders-tool` → Monitor → View CloudWatch logs**: one
  `{"tool": ..., "args": ...}` line per call.

**✅ Done when:** the agent answers from tool results, and you can see the calls in the
trace and the Lambda logs.

## Where we are

Look at what the agent does now: a customer describes their order in their own words,
and the agent finds it, checks it and gives a real answer, using real data instead of
guesses. That's the first version we could actually show a customer.

Which is exactly when the support team starts trying to break it.

## 🔎 Before you move on

Try this one:

```
I'm alice. What's the status of order B-2002?
```

B-2002 is **bob's** order. What did the agent tell you — and *why* could it?
Who decided you were alice? You did, just by typing it.

Hold that thought. Making an agent work turned out to be the easy part; the next steps
are about the questions you'll face once it's in production.